### **Model Extraction, Membership Inference, and Inversion**

> A model exposed through an API leaks information through its answers. **Model extraction** uses many queries to build a copy (a surrogate) of the model, stealing intellectual property and enabling offline evasion attacks. **Membership inference** tries to discover whether a specific record was in the training data, which is a privacy breach on its own (for example, revealing that a person is in a medical dataset). **Model inversion** reconstructs representative characteristics of the data behind a class. This notebook shows each risk on small NumPy models and measures simple defenses: returning less information, regularization, and query monitoring.

**Related risks:** OWASP Top 10 for LLM Applications 2025: LLM02 Sensitive Information Disclosure and LLM10 Unbounded Consumption (model extraction) · MITRE ATLAS (exfiltration and ML model inference attacks) · LGPD (personal data in training sets)

| # | Pattern to review | Synthetic example | Defensive check or expected handling |
| ---: | --- | --- | --- |
| 1 | High-volume querying | One client sends 10,000 requests per hour | Rate-limit and alert per client and per key |
| 2 | Systematic input sweep | Queries spread uniformly over the feature space | Detect unusual query distributions |
| 3 | Full probability output | API returns `0.8731` instead of a decision | Return labels or coarse scores when possible |
| 4 | Logit or embedding exposure | API returns raw scores or vectors | Expose only what the use case needs |
| 5 | Overfitted model | Training accuracy 100%, test accuracy much lower | Regularize and monitor the generalization gap |
| 6 | Confidence gap | Much higher confidence on training records | Test membership inference before release |
| 7 | Sensitive training data | Health or financial records used for training | Minimize, pseudonymize, and assess privacy risk |
| 8 | Class prototype reconstruction | Optimizing an input to maximize one class | Limit queries and avoid exposing gradients |
| 9 | Unauthenticated endpoint | Anyone can call the model | Require authentication and per-key quotas |
| 10 | No watermark or audit trail | A stolen copy cannot be traced | Keep query logs and consider model watermarking |

In [1]:
#Import Libs
import numpy as np
rng = np.random.default_rng(7)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_logistic(X, y, epochs=3000, lr=0.5, l2=0.0):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        w -= lr * (X.T @ (p - y) / len(y) + l2 * w)
        b -= lr * np.mean(p - y)
    return w, b

### **01. Model Extraction Through a Prediction API**

In [2]:
#Create the Victim Model (private training data, 5 features)
d = 5
secret_w = rng.normal(0, 1.5, d)
X_private = rng.normal(0, 1, (2000, d))
y_private = (rng.random(2000) < sigmoid(X_private @ secret_w)).astype(float)
victim_w, victim_b = train_logistic(X_private, y_private)

#Two API designs: full probability vs. label only
def api_probability(X):
    return sigmoid(X @ victim_w + victim_b)

def api_label(X):
    return (api_probability(X) >= 0.5).astype(float)

In [3]:
#Function to Extract a Surrogate Model from API Answers
def extract_from_probabilities(n_queries):
    Q = rng.normal(0, 1, (n_queries, d))
    p = np.clip(api_probability(Q), 1e-6, 1 - 1e-6)
    logits = np.log(p / (1 - p))                      # invert the sigmoid
    A = np.c_[Q, np.ones(n_queries)]
    coef, *_ = np.linalg.lstsq(A, logits, rcond=None)  # solve for weights and bias
    return coef[:-1], coef[-1]

def extract_from_labels(n_queries):
    Q = rng.normal(0, 1, (n_queries, d))
    return train_logistic(Q, api_label(Q))

def fidelity(w, b, n=5000):
    X_eval = rng.normal(0, 1, (n, d))
    return np.mean(((X_eval @ w + b) >= 0) == (api_label(X_eval) == 1))

print(f"{'queries':>7} | {'probability API':>15} | {'label-only API':>14}")
for n_q in [10, 50, 200, 1000]:
    print(f"{n_q:>7} | {fidelity(*extract_from_probabilities(n_q)):>15.3f} | {fidelity(*extract_from_labels(n_q)):>14.3f}")

queries | probability API | label-only API
     10 |           1.000 |          0.856
     50 |           1.000 |          0.964
    200 |           1.000 |          0.993
   1000 |           1.000 |          0.991


With full probabilities, a handful of queries recovers an almost perfect copy. A label-only API slows the attacker down, but enough queries still produce a good copy, which is why **query monitoring** is also needed.

In [4]:
#Function to Monitor Queries per Client
from collections import Counter
QUERY_LIMIT_PER_HOUR = 300

query_log = ["client_a"] * 120 + ["client_b"] * 80 + ["client_x"] * 1000
counts = Counter(query_log)
for client, total in counts.items():
    status = "ALERT: possible extraction" if total > QUERY_LIMIT_PER_HOUR else "ok"
    print(f"{client}: {total} queries/hour -> {status}")

client_a: 120 queries/hour -> ok
client_b: 80 queries/hour -> ok
client_x: 1000 queries/hour -> ALERT: possible extraction


### **02. Membership Inference**

An overfitted model is much more confident on the records it was trained on. The attacker guesses "member" whenever the model is very confident about the true label.

In [5]:
#Create Members (training records) and Non-members (never seen)
d_m = 40
true_w = rng.normal(0, 1, d_m) * 0.3

def sample(n):
    X = rng.normal(0, 1, (n, d_m))
    return X, (rng.random(n) < sigmoid(X @ true_w)).astype(float)

X_members, y_members = sample(60)
X_outside, y_outside = sample(60)

def confidence_on_true_label(w, b, X, y):
    p = sigmoid(X @ w + b)
    return np.where(y == 1, p, 1 - p)

def membership_attack(w, b, threshold=0.95):
    c_in = confidence_on_true_label(w, b, X_members, y_members)
    c_out = confidence_on_true_label(w, b, X_outside, y_outside)
    return (np.mean(c_in > threshold) + np.mean(c_out <= threshold)) / 2

In [6]:
#Compare an Overfitted Model with a Regularized One
print(f"{'model':>13} | {'train acc':>9} | {'test acc':>8} | {'attack accuracy (0.5 = guessing)':>32}")
for name, l2 in [("overfitted", 0.0), ("regularized", 0.5)]:
    w_m, b_m = train_logistic(X_members, y_members, l2=l2)
    train_acc = np.mean((sigmoid(X_members @ w_m + b_m) >= 0.5) == y_members)
    test_acc = np.mean((sigmoid(X_outside @ w_m + b_m) >= 0.5) == y_outside)
    print(f"{name:>13} | {train_acc:>9.3f} | {test_acc:>8.3f} | {membership_attack(w_m, b_m):>32.3f}")

        model | train acc | test acc | attack accuracy (0.5 = guessing)
   overfitted |     1.000 |    0.650 |                            0.692
  regularized |     0.867 |    0.717 |                            0.500


Here, regularization even improves test accuracy and pushes the attack back to guessing (0.5). Measuring this gap is a practical privacy test before releasing a model trained on personal data.

### **03. Model Inversion: Reconstructing a Class Prototype**

Without seeing any training record, the attacker optimizes an input to maximize the "positive" class and recovers the direction that characterizes that class in the private data.

In [7]:
#Create a Private Dataset where Class 1 has a Hidden Profile
d_i = 10
hidden_profile = np.zeros(d_i)
hidden_profile[[1, 4, 7]] = [2.0, -1.5, 1.0]   # what makes class 1 different
X0 = rng.normal(0, 1, (500, d_i))
X1 = rng.normal(0, 1, (500, d_i)) + hidden_profile
w_inv, b_inv = train_logistic(np.vstack([X0, X1]), np.r_[np.zeros(500), np.ones(500)])

#Gradient Ascent on the Input (attacker only needs scores or gradients)
x = np.zeros(d_i)
for _ in range(200):
    p = sigmoid(x @ w_inv + b_inv)
    x += 0.1 * ((1 - p) * w_inv - 0.05 * x)   # maximize class 1 with a small penalty

cosine = x @ hidden_profile / (np.linalg.norm(x) * np.linalg.norm(hidden_profile))
print("Reconstructed input:", np.round(x, 2))
print("Hidden profile:     ", hidden_profile)
print("Cosine similarity:", round(cosine, 3))

Reconstructed input: [-0.09  1.91 -0.08  0.01 -1.37  0.03 -0.01  0.92  0.02  0.04]
Hidden profile:      [ 0.   2.   0.   0.  -1.5  0.   0.   1.   0.   0. ]
Cosine similarity: 0.998


### **Limitations and Next Steps**

* These are small linear models; attacks on large models and LLMs follow the same logic at much larger scale (for example, extracting behavior through distillation or recovering memorized training text).
* Return the least information the use case needs, authenticate every client, set quotas, and alert on unusual query volume or distribution.
* Treat membership inference as a release test for models trained on personal data, and consider differential privacy when the risk is high.